In [1]:
import random
import time
import csv
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Set up Chrome
chrome_options = Options()
chrome_options.add_argument("--headless")  # Run without opening browser window
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--window-size=1920,1080")
driver = webdriver.Chrome(options=chrome_options)

def scrape_player(player_name):
    if not player_name.startswith("players/"):
        player_name = f"players/{player_name}"

    player_data = {}
    url = f'https://tankathon.com/{player_name}'
    driver.get(url)
    time.sleep(random.uniform(2, 5))
    
    try:
        # Close cookie popup if exists
        try:
            popup = 'CybotCookiebotDialogBodyButtonDecline'
            driver.find_element(By.ID, popup).click()
        except:
            pass

        # Player name
        name_element = WebDriverWait(driver, 10).until(
           EC.presence_of_element_located((By.XPATH, '//h1[@class="page-title"]'))
        )
        player_data['name'] = name_element.text

        # Pre-draft team
        try:
            team_element = WebDriverWait(driver, 3).until(
               EC.presence_of_element_located((By.XPATH, '//div[@class="team-link-section"]'))
            )
            player_data['pre-draft_team'] = team_element.text
        except:
            player_data['pre-draft_team'] = None

        # Info blocks
        try:
            info_section = driver.find_element(By.CLASS_NAME, "player-info")
            blocks = info_section.find_elements(By.CLASS_NAME, "data-block")
            for block in blocks:
                label = block.find_element(By.CLASS_NAME, "label").text.strip().lower()
                value = block.find_element(By.CLASS_NAME, "data").text.strip()
                mapping = {
                    "year": "school_year",
                    "position": "position",
                    "height": "height",
                    "weight": "weight",
                    "age at draft": "draft_age",
                    "birthdate": "birth_date",
                    "nation": "nation",
                    "hometown": "home_town",
                    "high school": "high_school"
                }
                if label in mapping:
                    player_data[mapping[label]] = value
        except:
            pass

        # Draft year
        try:
            draft_year = driver.find_element(By.CSS_SELECTOR, 'a.primary-hover[href*="/past_drafts/"]')
            player_data['draft_year'] = draft_year.text.strip().split()[0]
        except:
            player_data['draft_year'] = None

        # Draft position
        try:
            draft_position = driver.find_element(By.CSS_SELECTOR, 'div.number a.primary-hover')
            player_data['draft_position'] = draft_position.text.strip().split()[0]
        except:
            player_data['draft_position'] = None

        # Combine stats
        try:
            stat_blocks = driver.find_elements(By.CLASS_NAME, "combine-stat")
            for block in stat_blocks:
                label_elem = block.find_element(By.CLASS_NAME, "combine-stat-label")
                value_elem = block.find_element(By.CLASS_NAME, "combine-stat-value")
                label_text = label_elem.text.strip().lower().replace(" ", "_")
                player_data[f"combine_{label_text}"] = value_elem.text.strip()
        except:
            pass

        # Find all stat-row divs (there should be two - per game and per 36)
        try:
            stat_rows = driver.find_elements(By.CLASS_NAME, "stat-row")
            
            for row_idx, stat_row in enumerate(stat_rows):
                stat_containers = stat_row.find_elements(By.CLASS_NAME, "stat-container")
                
                # Determine if this is per 36 stats (usually the second row)
                is_per_36 = row_idx > 0
                suffix = "/36" if is_per_36 else ""
                
                for container in stat_containers:
                    try:
                        label_elem = container.find_element(By.CLASS_NAME, "stat-label")
                        value_elem = container.find_element(By.CLASS_NAME, "stat-data")
                        label_text = label_elem.text.strip()
                        value_text = value_elem.text.strip()
                        
                        # Create the key - add /36 suffix for second stat row
                        key = label_text + suffix
                        player_data[key] = value_text
                    except:
                        continue
        except Exception as e:
            print(f"[DEBUG] Could not parse stat rows: {e}")

        # Look for stat tables - this might contain season-by-season stats
        try:
            tables = driver.find_elements(By.TAG_NAME, "table")
            for table_idx, table in enumerate(tables):
                try:
                    # Get headers
                    headers = []
                    thead = table.find_element(By.TAG_NAME, "thead")
                    header_cells = thead.find_elements(By.TAG_NAME, "th")
                    headers = [h.text.strip() for h in header_cells]
                    
                    # Get first data row (usually the most recent season)
                    tbody = table.find_element(By.TAG_NAME, "tbody")
                    rows = tbody.find_elements(By.TAG_NAME, "tr")
                    
                    if rows and headers:
                        first_row = rows[0]
                        cells = first_row.find_elements(By.TAG_NAME, "td")
                        
                        for idx, cell in enumerate(cells):
                            if idx < len(headers) and headers[idx]:
                                col_name = f"table_{headers[idx]}"
                                player_data[col_name] = cell.text.strip()
                except:
                    continue
        except:
            pass

    except Exception as e:
        print(f"[DEBUG] error scraping {url}: {e}")
        player_data['name'] = 'Info not found'
        
    player_data['url'] = url

    return player_data


def get_players_from_draft(year):
    """Scrape a draft page and return all player URLs"""
    url = f"https://tankathon.com/past_drafts/{year}"
    driver.get(url)
    time.sleep(random.uniform(2, 4))

    # Close cookie popup if exists
    try:
        popup = 'CybotCookiebotDialogBodyButtonDecline'
        driver.find_element(By.ID, popup).click()
    except:
        pass

    # Grab all player links
    player_links = driver.find_elements(By.CSS_SELECTOR, "a.primary-hover[href*='players/']")
    player_names = [link.get_attribute("href").split("tankathon.com/")[-1] for link in player_links]

    # Remove duplicates
    return list(set(player_names))


# Main execution - only 2025
all_players = []

print("Scraping 2025 draft class...")
players = get_players_from_draft(2025)
print(f"Found {len(players)} players")

for idx, p in enumerate(players, 1):
    print(f"Scraping player {idx}/{len(players)}: {p}")
    data = scrape_player(p)
    all_players.append(data)
    print(f"  Captured {len(data)} fields for {data.get('name', 'unknown')}")

# Define the field order to match Tankathon's layout
field_order = [
    'name',
    'pre-draft_team',
    'draft_year',
    'draft_position',
    'position',
    'height',
    'weight',
    'school_year',
    'draft_age',
    'birth_date',
    'nation',
    'home_town',
    'high_school',
    # Combine stats
    'combine_height_w/o_shoes',
    'combine_height_w/_shoes',
    'combine_weight',
    'combine_wingspan',
    'combine_standing_reach',
    'combine_body_fat',
    'combine_hand_length',
    'combine_hand_width',
    'combine_bench_press',
    'combine_agility',
    'combine_sprint',
    'combine_vertical_(max)',
    'combine_vertical_(max_reach)',
    'combine_vertical_(no_step)',
    'combine_vertical_(no_step_reach)',
    # Per game stats
    'G',
    'MP',
    'FGM-FGA',
    'FG%',
    '3PM-3PA',
    '3P%',
    'FTM-FTA',
    'FT%',
    'REB',
    'AST',
    'BLK',
    'STL',
    'TO',
    'PF',
    'PTS',
    # Per 36 stats
    'G/36',
    'MP/36',
    'FGM-FGA/36',
    'FG%/36',
    '3PM-3PA/36',
    '3P%/36',
    'FTM-FTA/36',
    'FT%/36',
    'REB/36',
    'AST/36',
    'BLK/36',
    'STL/36',
    'TO/36',
    'PF/36',
    'PTS/36',
    'url'
]

# Collect all unique fieldnames across players
all_fieldnames = set()
for player in all_players:
    all_fieldnames.update(player.keys())

# Start with defined order, then add any extra fields found
fieldnames = []
for field in field_order:
    if field in all_fieldnames:
        fieldnames.append(field)
        all_fieldnames.remove(field)

# Add any remaining fields that weren't in our predefined order
fieldnames.extend(sorted(list(all_fieldnames)))

print(f"\nTotal columns captured: {len(fieldnames)}")

# Write to CSV
output_file = "tankathon_2025_draft.csv"
with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_players)

print(f"\nData saved to {output_file}")
driver.quit()

Scraping 2025 draft class...
Found 83 players
Scraping player 1/83: players/lachlan-olbrich
  Captured 67 fields for Lachlan Olbrich
Scraping player 2/83: players/chaz-lanier
  Captured 67 fields for Chaz Lanier
Scraping player 3/83: players/khaman-maluach
  Captured 67 fields for Khaman Maluach
Scraping player 4/83: players/alijah-martin
  Captured 67 fields for Alijah Martin
Scraping player 5/83: players/brooks-barnhizer
  Captured 61 fields for Brooks Barnhizer
Scraping player 6/83: players/drake-powell
  Captured 67 fields for Drake Powell
Scraping player 7/83: players/cedric-coward
  Captured 67 fields for Cedric Coward
Scraping player 8/83: players/maxime-raynaud
  Captured 67 fields for Maxime Raynaud
Scraping player 9/83: players/andre-curbelo
  Captured 61 fields for Andre Curbelo
Scraping player 10/83: players/ace-bailey
  Captured 67 fields for Ace Bailey
Scraping player 11/83: players/wooga-poplar
  Captured 61 fields for Wooga Poplar
Scraping player 12/83: players/tyrese-p